# 05 — Full Kaggle Pipeline

**USE THIS FOR A QUICK SMOKE TEST ONLY.**  
For full training runs, use the individual notebooks (05A → 05B × 5 → 05C → 05D) which manage disk better.

This notebook runs everything sequentially in one session:
data prep → classical baselines → train 5 models → evaluate → package artifacts.

**YOLO is excluded** — its image dataset would exceed the 19.5 GB limit when combined with prepared data. Run 05D separately after saving 05A's output.

### Datasets to attach
| Dataset | Role |
|---|---|
| `brats20-dataset-training-validation` by awsaf49 | Raw NIfTI (read-only, free) |
| Your code zip (`brats-seg-code`) | Source code |

In [ ]:
# ── RUNTIME SWITCHES ──────────────────────────────────────────────────────
GITHUB_REPO = 'https://github.com/yasmineelsayeddd/brain-tumor-segmentation.git'

# Set to 2 for a 5-minute sanity check. None = real training.
EPOCH_OVERRIDE = 2

# Subset of test slices for classical baselines (None = all, slow)
CLASSICAL_LIMIT = 200

# Disk-safe settings — do NOT change these
KEEP_EMPTY_RATIO = 0.0
IMAGE_DTYPE      = 'float16'
MIN_TUMOR_PIXELS = 20
SEED             = 42

SEGMENTATION_CONFIGS = [
    'configs/default.yaml',
    'configs/unet_best_fusion_loss_aug.yaml',
    'configs/unetpp.yaml',
    'configs/attention_unet.yaml',
    'configs/uncertainty_unet.yaml',
]

In [ ]:
import os, shutil, subprocess, sys, json, yaml
from pathlib import Path

def disk_gb(path='/'):
    st = os.statvfs(path)
    return round(st.f_bavail * st.f_frsize / 1e9, 2)

def folder_gb(path):
    p = Path(path)
    if not p.exists(): return 0.0
    return round(sum(f.stat().st_size for f in p.rglob('*') if f.is_file()) / 1e9, 2)

print(f'Free disk at start: {disk_gb()} GB  (limit 19.5 GB)')

In [ ]:
# ── 1. Clone code repo from GitHub ─────────────────────────────────────────
PROJECT = Path('/kaggle/working/project')
if PROJECT.exists():
    shutil.rmtree(PROJECT)

subprocess.check_call(['git', 'clone', '--depth', '1', GITHUB_REPO, str(PROJECT)])
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

def run(cmd):
    subprocess.check_call(cmd, cwd=str(PROJECT))

print('Repo cloned to:', PROJECT)

In [ ]:
# ── 2. Install missing packages ────────────────────────────────────────────
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nibabel', 'albumentations'])
import torch
print(f'CUDA: {torch.cuda.is_available()}  |  Free disk: {disk_gb()} GB')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── 3. Find raw BraTS ──────────────────────────────────────────────────────
RAW_BRATS = next(
    (p for p in [
        Path('/kaggle/input/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'),
        Path('/kaggle/input/brats2020-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'),
    ] if p.exists()),
    None
)
if RAW_BRATS is None:
    hits = list(Path('/kaggle/input').rglob('MICCAI_BraTS2020_TrainingData'))
    RAW_BRATS = hits[0] if hits else None
if RAW_BRATS is None:
    raise FileNotFoundError('Attach brats20-dataset-training-validation.')

print(f'Raw BraTS: {RAW_BRATS}')
print(f'Patients:  {len(sorted(p for p in RAW_BRATS.iterdir() if p.is_dir()))}')

In [ ]:
# ── 4. Prepare 2D slices (float16, tumor-only) ─────────────────────────────
DATA_2D    = Path('/kaggle/working/brats2020_2d')
SPLIT_FILE = PROJECT / 'configs/splits/default.json'

if not (DATA_2D / 'metadata.json').exists():
    print(f'Preparing slices — expected ~15 min. Free disk before: {disk_gb()} GB')
    run([
        sys.executable, '-m', 'scripts.prepare_data',
        '--input',            str(RAW_BRATS),
        '--output',           str(DATA_2D),
        '--keep-empty-ratio', str(KEEP_EMPTY_RATIO),
        '--image-dtype',      IMAGE_DTYPE,
        '--min-tumor-pixels', str(MIN_TUMOR_PIXELS),
        '--seed',             str(SEED),
    ])
else:
    print('Slices already prepared.')

with open(DATA_2D / 'metadata.json') as f:
    meta = json.load(f)
print(f'Slices: {meta["num_slices"]:,}  |  dtype: {meta.get("image_dtype")}  |  size: {folder_gb(DATA_2D)} GB')
print(f'Free disk after prep: {disk_gb()} GB')

In [ ]:
# ── 5. Split ───────────────────────────────────────────────────────────────
SPLIT_FILE.parent.mkdir(parents=True, exist_ok=True)
if not SPLIT_FILE.exists():
    run([
        sys.executable, '-m', 'scripts.make_splits',
        '--data-root', str(DATA_2D),
        '--output',    str(SPLIT_FILE),
        '--seed',      str(SEED),
    ])
with open(SPLIT_FILE) as f:
    split = json.load(f)
print({k: len(v) for k, v in split.items()})

In [ ]:
# ── 6. Runtime config builder ──────────────────────────────────────────────
RT_DIR = Path('/kaggle/working/rt_configs')
RT_DIR.mkdir(parents=True, exist_ok=True)

def make_rt_config(cfg_path: str) -> str:
    with open(PROJECT / cfg_path) as f:
        cfg = yaml.safe_load(f)
    cfg['data']['data_root']  = str(DATA_2D)
    cfg['data']['split_file'] = str(SPLIT_FILE)
    cfg['checkpoint_dir']     = '/kaggle/working/checkpoints'
    cfg['experiment']['output_dir'] = '/kaggle/working/outputs'
    if EPOCH_OVERRIDE is not None:
        cfg['training']['epochs'] = int(EPOCH_OVERRIDE)
        cfg['training']['early_stopping_patience'] = max(2, min(int(EPOCH_OVERRIDE), 15))
    out = RT_DIR / Path(cfg_path).name
    with open(out, 'w') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return str(out)

print('Runtime configs ready.')

In [ ]:
# ── 7. Classical baselines ─────────────────────────────────────────────────
cmd = [sys.executable, '-m', 'scripts.classical_baselines',
       '--config', make_rt_config('configs/default.yaml'), '--split', 'test']
if CLASSICAL_LIMIT:
    cmd += ['--limit', str(CLASSICAL_LIMIT)]
print('Running classical baselines …')
run(cmd)

In [ ]:
# ── 8. Train all 5 models ──────────────────────────────────────────────────
for cfg_path in SEGMENTATION_CONFIGS:
    rt = make_rt_config(cfg_path)
    with open(rt) as f:
        cfg = yaml.safe_load(f)
    exp  = cfg['experiment']['name']
    ckpt = Path('/kaggle/working/checkpoints') / f'{exp}_best.pth'
    if ckpt.exists():
        print(f'[SKIP] {exp} — checkpoint already exists')
        continue
    print(f'\n=== Training {exp} (epochs={cfg["training"]["epochs"]}) ===')
    print(f'Free disk: {disk_gb()} GB')
    run([sys.executable, '-m', 'scripts.train', '--config', rt])
    print(f'Free disk after: {disk_gb()} GB')

In [ ]:
# ── 9. Evaluate all checkpoints ────────────────────────────────────────────
for cfg_path in SEGMENTATION_CONFIGS:
    rt = make_rt_config(cfg_path)
    with open(rt) as f:
        cfg = yaml.safe_load(f)
    exp  = cfg['experiment']['name']
    ckpt = Path('/kaggle/working/checkpoints') / f'{exp}_best.pth'
    if not ckpt.exists():
        print(f'[SKIP] {exp} — no checkpoint')
        continue
    print(f'\n=== Evaluating {exp} ===')
    run([
        sys.executable, '-m', 'scripts.evaluate',
        '--config',     rt,
        '--checkpoint', str(ckpt),
        '--split',      'test',
    ])

In [ ]:
# ── 10. Results summary ────────────────────────────────────────────────────
import pandas as pd, matplotlib.pyplot as plt

rows = []
for cfg_path in SEGMENTATION_CONFIGS:
    with open(RT_DIR / Path(cfg_path).name) as f:
        cfg = yaml.safe_load(f)
    exp = cfg['experiment']['name']
    mj = Path('/kaggle/working/outputs') / exp / 'test_metrics.json'
    if mj.exists():
        with open(mj) as f:
            m = json.load(f)
        rows.append({'model': exp, **{k: v for k, v in m.items() if isinstance(v, float)}})

if rows:
    df = pd.DataFrame(rows).set_index('model')
    print(df[['mean_dice', 'mean_iou', 'pixel_accuracy']].round(4).to_string())
    df['mean_dice'].plot(kind='barh', figsize=(9, 5), color='steelblue', edgecolor='white')
    plt.xlabel('Mean Dice'); plt.title('Model Comparison'); plt.tight_layout(); plt.show()
else:
    print('No results yet.')

In [ ]:
# ── 11. Ablation manifest + package artifacts ──────────────────────────────
run([
    sys.executable, '-m', 'scripts.ablation_manifest',
    '--output', '/kaggle/working/outputs/ablation_manifest.csv',
])

EXPORT = Path('/kaggle/working/artifacts')
if EXPORT.exists(): shutil.rmtree(EXPORT)
EXPORT.mkdir()
for name in ['checkpoints', 'outputs']:
    src = Path('/kaggle/working') / name
    if src.exists(): shutil.copytree(src, EXPORT / name)

archive = shutil.make_archive('/kaggle/working/brats_artifacts', 'zip', EXPORT)
print('Packaged:', archive)
print(f'Final free disk: {disk_gb()} GB')